## If you haven't already; please create a .env file in the base level of this directory (one step above this folder)
## use environment variables TIF_PATH and CONTENT_PATH: 
### TIF_PATH is where your bat image tif files are located, CONTENT_PATH is where they will be stored after processing
## example: 
* TIF_PATH=/mnt/d/
* CONTENT_PATH=/home/elliot/BatGutsImageClassification2/

In [7]:
env_USE_GOOGLE_COLAB = False
env_USE_NVIDIA = False
env_CONTENT_PATH = ""
env_TIF_PATH = ""
env_IMAGE_SHAPE_X = 0
env_IMAGE_SHAPE_Y = 0
import sys
from settings import load_BatGutsSettings, show_python_version, corr2
show_python_version()

S = load_BatGutsSettings()
env_CONTENT_PATH = S.zContentPath
env_TIF_PATH = S.zTifPath
env_USE_GOOGLE_COLAB = S.useGoogleColab
env_USE_NVIDIA = S.useNvidia
env_IMAGE_SHAPE_X = S.imageShapeX
env_IMAGE_SHAPE_Y = S.imageShapeY
if len(env_CONTENT_PATH) == 0:
    sys.exit("Exiting the script because setings were not processed")

Python version: 3.10.12
Loading settings from /home/elliot/BatGutsImageClassification2/jupyternb/.env
my_CONTENT_PATH=/home/elliot/BatGutsImageClassification2/
my_TIF_PATH=/mnt/d/


In [8]:
#this code generates the folder structure for storing the processed image files based on your CONTENT_PATH variable
import os

images_path = os.path.join(env_CONTENT_PATH, 'images')
if os.path.exists(images_path):
    print(f"The directory {images_path} exists.")
else:
     print(f"The directory {images_path} does not exist.")
     try:
        os.makedirs(os.path.join(images_path, 'Bats_mixed_images_RGB_256'))
        print(f"Created directory {os.path.join(images_path, 'Bats_mixed_images_RGB_256')}")
     except OSError as e:
        print(f"Error creating directory: {e}")
     bat_images_path = os.path.join(images_path, 'Bats_mixed_images_RGB_256')
     try:
        os.makedirs(os.path.join(bat_images_path, 'Plants'))
        print(f"Created directory {os.path.join(bat_images_path, 'Plants')}")
     except OSError as e:
        print(f"Error creating directory: {e}")
     try:
        os.makedirs(os.path.join(bat_images_path, 'Blood'))
        print(f"Created directory {os.path.join(bat_images_path, 'Blood')}")
     except OSError as e:
        print(f"Error creating directory: {e}")
     try:
        os.makedirs(os.path.join(bat_images_path, 'Insects'))
        print(f"Created directory {os.path.join(bat_images_path, 'Insects')}")
     except OSError as e:
        print(f"Error creating directory: {e}")

The directory /home/elliot/BatGutsImageClassification2/images exists.


In [9]:
import numpy as np
from matplotlib import pyplot as plt
import tifffile as tiff
import numpy as np
from PIL import Image as im
from PIL import  ImageOps
import cv2
# plt.imshow(image)
import time
import os
import glob

In [10]:
'''
processes TIFF images from a specified directory and 
saves the red, green, and blue channels of each image as separate JPEG files.
'''
def Save_images_in_r_g_b(tiffDir):
  if "insect" in tiffDir:
    category = "Insects"
  elif "blood" in  tiffDir:
    category = "Blood"
  else:
    category = "Plants"
  for main_path in glob.glob(tiffDir+"*/"):
    # print(main_path)
    paths=glob.glob(os.path.join(main_path,"*.tif"))
    for image_path in paths:
      if "Plane" in image_path or "gutMessy" in image_path:
        continue
        #we skipping images w/ multiple planes for now as well as the messy areas
      print(paths.index(image_path))
      base_image_path = os.path.basename(image_path)
      # print(str(base_image_path[0:-4]))
      large_image = tiff.imread(image_path)
      newpath = env_CONTENT_PATH + '/images/Bats_mixed_images_RGB_256/'+category+'/'+ str(base_image_path[0:-4])
      print("newPath="+newpath)
      r, g, b = np.moveaxis(large_image, -1, 0)
      tiff.imwrite(newpath+"__"+"red"+".jpeg",r)
      tiff.imwrite(newpath+"__"+"green"+".jpeg",g)
      tiff.imwrite(newpath+"__"+"blue"+".jpeg",b)


      print("Completed for "+" "+str(image_path[40:-4]) )
      time.sleep(0.2)

    print("Completed")

In [11]:
'''
this function is designed to resize all JPEG images in a specific directory (related to b
at images) to 256x256 pixels, overwriting the original files with their resized versions. 
It includes error handling to manage potential issues with individual images or the overall process.
'''
def resizing_image(category):
  try:
    print(category)
    for main_path in glob.glob( env_CONTENT_PATH+"/images/Bats_mixed_images_RGB_256/"+ str(category)+"/"):
      paths=glob.glob(os.path.join(main_path,"*.jpeg"))
      for image_path in paths:
        base_image_path = os.path.basename(image_path)
        print( str(paths.index(image_path)) + " " + base_image_path )
        try:
          image = cv2.imread(image_path)
        except Exception  as e:
          print(image_path)
          print(e)
          continue
        resized_image = cv2.resize(image,(256,256))
        cv2.imwrite(image_path, resized_image)
    print('completed')
  except Exception  as e:
    print(e)







In [12]:
'''
The purpose of setting OPENCV_IO_MAX_IMAGE_PIXELS is to increase the maximum allowed size of images 
that OpenCV can process. By default, OpenCV has a limit on the maximum number of pixels in an image 
it can handle to prevent accidental loading of extremely large files that could consume too much memory.
By setting this value to 2^40, the code is allowing OpenCV to work with exceptionally large images, 
up to about 1.1 trillion pixels. This is useful when dealing with very high-resolution images or large 
panoramas that exceed OpenCV's default limits.
It's important to note that while this allows processing of larger images, 
it also increases the risk of running out of memory if your system doesn't have enough RAM to 
handle such large images. Use this setting cautiously and ensure your system has sufficient resources to 
handle the images you intend to process. 
'''

import os
x = pow(2,40)
print( "Setting OPENCV_IO_MAX_IMAGE_PIXELS to " + str( x ))
os.environ["OPENCV_IO_MAX_IMAGE_PIXELS"] = str(x)
import cv2

Setting OPENCV_IO_MAX_IMAGE_PIXELS to 1099511627776


In [13]:
"""
Use this to download the folders from your TIF location
If not already set up, create a .env file in the base of your directory
Then use TIF_PATH and CONTENT_PATH to set up where to load the TIF files and where to place them respectively
"""
cats = ["fruit", "nectar", "blood", "insect"]
for folder in glob.glob(env_TIF_PATH+"*/"):
    if any(e in folder for e in cats):
        if "jamaicencis" in folder or "literatus" in folder:
            continue
        Save_images_in_r_g_b(folder)

In [14]:
resizing_image('Blood')
resizing_image('Plants')
resizing_image('Insects')

Blood
completed
Plants
0 T2023-69_Artibeus_literatus_LY23-1-8C_fruit_middle_LY24-1-8C AB 2-2__blue.jpeg
1 T2023-21L_Uroderma_bilobatum_LY24-1-6E_fruit_distal_AB_LY24-1-6E AB 1-2__red.jpeg
2 T2023-23L_Platyrrhinus_helleri-LY24-1-5A_fruit_stomach_AB_LY24-1-5A AB 1-2__red.jpeg
3 T2023-14L_Artibeus_jamaicencis_LY24-1-1B_fruit_proximal_AB_LY24-1-1B AB 1-2__red.jpeg
4 T2023-21L_Uroderma_bilobatum_LY24-1-6D_fruit_threeQuarters_AB-PAS_LH24-1-6B AB-PAS 1-2__blue.jpeg
5 T2023-23L_Platyrrhinus_helleri-LY24-1-5E_fruit_distal_AB_LY24-1-5E AB 1-2__blue.jpeg
6 T2023-14L_Artibeus_jamaicencis_LY24-1-1C_fruit_middle_AB_LY24-1-1C AB 1-2__red.jpeg
7 LY24-1-8D AB-PAS 1-2__green.jpeg
8 T2023-14L_Artibeus_jamaicencis_LY24-1-1B_fruit_proximal_AB_LY24-1-1B AB 1-2__blue.jpeg
9 T2023-14L_Artibeus_jamaicencis_LY24-1-1E_fruit_distal_AB_LY24-1-1E AB 1-2__blue.jpeg
10 T2023-23L_Platyrrhinus_helleri-LY24-1-5C_fruit_middle_AB_LY24-1-5C AB 1-2__blue.jpeg
11 T2023-14L_Artibeus_jamaicencis_LY24-1-1E_fruit_distal_HE_LY24-